# Solutions — Architecture

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

Each code cell here is self-contained: it redefines the helpers it needs, because a solutions
notebook has its own scope and you may run any cell on its own.

### LESSON 65 — Exercise

In [ ]:
// L65 solution — a third feature, and folders touched

const l65sFiles = [
  "src/components/ExpenseForm.jsx",
  "src/components/ExpenseList.jsx",
  "src/components/ExpenseRow.jsx",
  "src/components/ReportChart.jsx",
  "src/components/ReportFilters.jsx",
  "src/components/BudgetCard.jsx",
  "src/components/Button.jsx",
  "src/hooks/useExpenses.js",
  "src/hooks/useReport.js",
  "src/hooks/useBudget.js",
  "src/services/expenses.js",
  "src/services/report.js",
  "src/services/budget.js",
  "src/utils/validateExpense.js",
  "src/utils/validateBudget.js",
  "src/utils/formatMoney.js",
];

const l65sFolderOf = (p) => p.split("/").slice(0, -1).join("/");
const l65sNameOf = (p) => p.split("/").pop();

function l65sFeatureOf(path) {
  const name = l65sNameOf(path);
  if (/expense/i.test(name)) return "src/features/expenses";
  if (/report/i.test(name)) return "src/features/reports";
  if (/budget/i.test(name)) return "src/features/budgets";
  return "src/shared";
}

const l65sByFeature = l65sFiles.map((p) => `${l65sFeatureOf(p)}/${l65sNameOf(p)}`);

// 2. one function, either layout
function l65sFoldersTouched(files, keyword) {
  const hit = files.filter((p) => new RegExp(keyword, "i").test(l65sNameOf(p)));
  return new Set(hit.map(l65sFolderOf)).size;
}

console.log("feature   by type   by feature");
for (const keyword of ["expense", "report", "budget"]) {
  const byType = l65sFoldersTouched(l65sFiles, keyword);
  const byFeature = l65sFoldersTouched(l65sByFeature, keyword);
  console.log(`${keyword.padEnd(10)}${String(byType).padEnd(10)}${byFeature}`);
}

// 3. the two that refused to be classified
console.log(
  "\nunclassifiable:",
  l65sByFeature.filter((p) => l65sFolderOf(p) === "src/shared").map(l65sNameOf),
);

// They are Button.jsx and formatMoney.js, and they landed in src/shared.
//   The rule that put them there: they are imported by more than one feature, so they belong
//   to none of them.
//   The rule for moving a feature file out to join them: move it when the SECOND feature
//   imports it — not when it merely looks generic.

**Common mistake:** matching the keyword against the whole path instead of the file name. Under
the by-feature layout the path already contains `features/expenses/`, so `/expense/i` against
the full path matches every file in the folder and the count comes out wrong — and, worse, it
comes out wrong in the direction that flatters the answer you were hoping for. Match on the file
name, which is the same string under both layouts.

Notice what the table shows: **by feature the number is 1 for every feature**, and it stays 1 as
the app grows. By type it grows with the number of file kinds, which is the number that keeps
increasing as a project matures — add tests and styles per component and it is six, not four.

### LESSON 65 — Mini challenge

In [ ]:
// L65 solution — what actually belongs in shared/

const l65sImports = {
  "formatMoney.js":     ["expenses", "reports"],
  "validateExpense.js": ["expenses"],
  "Button.jsx":         ["expenses", "reports", "budgets"],
  "expenseColours.js":  ["expenses"],
  "useReport.js":       ["reports"],
  "dateRange.js":       ["reports", "budgets"],
};

function l65sShouldBeShared(imports) {
  const shared = [];
  const stays = [];
  for (const [module, features] of Object.entries(imports)) {
    (new Set(features).size > 1 ? shared : stays).push(module);
  }
  return { shared, stays };
}

const l65sVerdict = l65sShouldBeShared(l65sImports);
console.log("belongs in shared/:", l65sVerdict.shared);
console.log("stays in its feature:", l65sVerdict.stays);

// 2. expenseColours.js sounds generic and is used by one feature.
//    Moving it to shared/ costs three things: the expenses folder stops being the whole story
//    for an expenses change; the next person assumes other features depend on it and stops
//    editing it freely; and the file acquires an implied general-purpose API it does not have.
//    Move it the day a second feature imports it — that import is the evidence.
//
// 3. What the rule protects you from: designing for imaginary callers. "Anything reusable goes
//    in shared" is unfalsifiable — everything is reusable in principle — so shared/ becomes the
//    junk drawer, and the modules in it get generalised for callers that never arrive. Waiting
//    for the second import means every file in shared/ has at least two real callers whose
//    needs you can actually read.

### LESSON 66 — Exercise

In [ ]:
// L66 solution — recomputing the owner as the tree changes

const l66sTree = {
  name: "App",
  children: [
    {
      name: "Header",
      children: [
        { name: "MenuButton", children: [] },
        { name: "MenuList", children: [{ name: "MenuItem", children: [] }] },
      ],
    },
    {
      name: "Directory",
      children: [
        { name: "SearchBox", children: [] },
        {
          name: "Results",
          children: [{ name: "ResultRow", children: [] }, { name: "EmptyState", children: [] }],
        },
        // 1. the detail panel arrives
        { name: "DetailPanel", children: [{ name: "DetailField", children: [] }] },
      ],
    },
    { name: "Footer", children: [] },
  ],
};

function l66sPathTo(node, target, trail = []) {
  const here = [...trail, node.name];
  if (node.name === target) return here;
  for (const child of node.children) {
    const found = l66sPathTo(child, target, here);
    if (found) return found;
  }
  return null;
}

function l66sOwner(readers) {
  const paths = readers.map((n) => l66sPathTo(l66sTree, n));
  const common = [];
  for (let i = 0; i < paths[0].length; i += 1) {
    if (paths.every((p) => p[i] === paths[0][i])) common.push(paths[0][i]);
    else break;
  }
  return common[common.length - 1];
}

function l66sFind(node, name) {
  if (node.name === name) return node;
  for (const child of node.children) {
    const found = l66sFind(child, name);
    if (found) return found;
  }
  return null;
}

const l66sSize = (node) => 1 + node.children.reduce((t, c) => t + l66sSize(c), 0);

function l66sReport(label, readers) {
  const owner = l66sOwner(readers);
  console.log(
    `${label.padEnd(12)} readers: ${readers.join(", ").padEnd(30)} owner: ${owner.padEnd(10)} re-renders: ${l66sSize(l66sFind(l66sTree, owner))}`,
  );
  return owner;
}

l66sReport("selectedId", ["ResultRow", "DetailPanel"]);      // 1
l66sReport("isMenuOpen", ["MenuButton", "MenuList"]);        // 2, before
l66sReport("isMenuOpen+", ["MenuButton", "MenuList", "MenuItem"]);  // 2, after
l66sReport("theme", ["MenuList", "EmptyState", "Footer"]);   // 3

**1.** `selectedId` lives in `Directory` — the closest common parent of `ResultRow` and
`DetailPanel` — and changing the selection re-renders `Directory`'s subtree, seven components.
Note what it is *not*: `App`. A selection is a directory concern, and putting it in `App` would
re-render the header and footer every time you clicked a row.

**2.** The owner does **not** move. `MenuItem` is inside `MenuList`, which is already inside
`Header`, so the closest common parent is unchanged. This is the useful half of the exercise:
adding a reader only moves the owner when the new reader sits outside the current subtree.

**3.** `theme` is read by `MenuList`, `EmptyState` and `Footer`, which are in three different
branches, so the owner is `App` — React's **option 1**, the common parent, and it happens to be
the root. That is the honest answer to "where does it live", and it is also exactly the shape
LESSON 56 identified as prop drilling: the value must reach three distant leaves, and every
component in between would have to accept and forward a prop it does not use. Context is the
usual answer for a theme because the *owner* is right but the *delivery* is the problem — the
state still lives in one place, and Context only removes the forwarding.

**Common mistake:** concluding that because the owner is `App`, everything in the app must
re-render on a theme change, and therefore Context is a performance fix. It is not — Context
changes how the value gets there, not who owns it. The re-render count is the same. Performance
is topic 23, and it starts by measuring rather than guessing.

### LESSON 66 — Mini challenge

In [ ]:
// L66 solution — state, derived, ref, or the URL?

function l66sClassify(value) {
  const answers = {
    "typed search text":
      ["state", "it changes over time, the user types it, and nothing else can produce it"],
    "number of results shown":
      ["derived", "it is filteredResults.length — computable during render from state you have"],
    "debounce timer id":
      ["ref", "it must survive renders and must NOT cause one; a setState here would loop"],
    "department filter":
      ["URL", "someone might send a link to it — and the Back button should undo it"],
  };
  return answers[value];
}

for (const value of [
  "typed search text",
  "number of results shown",
  "debounce timer id",
  "department filter",
]) {
  const [where, why] = l66sClassify(value);
  console.log(`${value.padEnd(26)} ${where.padEnd(9)} ${why}`);
}

// The two interesting ones:
//
// (2) the result count is the one wrongly made into state. Doing that gives you two sources of
//     truth — the list and its count — and they drift the first time a code path updates one
//     without the other. A derived const cannot drift, because it does not exist between
//     renders (L29).
//
// (4) the department filter is the one wrongly left out of the URL. Kept in useState, the
//     filtered view has no address: the user cannot send it to a colleague, cannot bookmark it,
//     and pressing Back does not undo the filter — it leaves the page entirely, because the
//     filter never created a history entry (L61). Anything a user might want to link to is a
//     URL question before it is a state question.

### LESSON 67 — Exercise

In [ ]:
// L67 solution — auditing and redesigning a prop API

const l67sProps = {
  user: { id: 7, name: "Ada Lovelace", email: "ada@example.com" },
  showEmail: true,
  showPhone: true,
  hideAvatar: true,
  isCompact: true,
  isLarge: true,
  red: true,
  onClick: () => {},
};

const l67sAppearance = ["red", "blue", "green", "bold", "big", "small"];
const l67sExclusive = [["isCompact", "isLarge"]];

function l67sAudit(props) {
  const findings = [];
  const keys = Object.keys(props);

  for (const [a, b] of l67sExclusive) {
    if (props[a] && props[b]) findings.push(`${a} and ${b} cannot both be true`);
  }

  const shows = keys.filter((k) => /^show[A-Z]/.test(k));
  const hides = keys.filter((k) => /^hide[A-Z]/.test(k));
  if (shows.length && hides.length) {
    findings.push(`mixed polarity: ${shows.join(", ")} alongside ${hides.join(", ")}`);
  }

  for (const key of keys) {
    if (l67sAppearance.includes(key)) {
      findings.push(`"${key}" names an appearance, not a meaning`);
    }
  }

  return findings;
}

console.log("findings:");
for (const f of l67sAudit(l67sProps)) console.log(" -", f);

// 2. the redesign
//
//    <UserCard
//      user={user}
//      size="compact"                 // one prop, three values: compact | default | large
//      tone="danger"                  // meaning, not colour
//      fields={["email", "phone"]}    // one list, no show/hide pairs, avatar included or not
//      onSelect={…}                   // says what happened, not which input fired
//    />

const l67sSizes = ["compact", "default", "large"];
const l67sTones = ["neutral", "danger"];
const l67sFields = ["avatar", "email", "phone"];

function l67sValid(props) {
  if (!l67sSizes.includes(props.size ?? "default")) return false;
  if (!l67sTones.includes(props.tone ?? "neutral")) return false;
  const fields = props.fields ?? [];
  return Array.isArray(fields) && fields.every((f) => l67sFields.includes(f));
}

console.log("\nnew API:");
console.log(" compact + danger + [email]:", l67sValid({ size: "compact", tone: "danger", fields: ["email"] }));
console.log(" nothing but defaults      :", l67sValid({}));
console.log(" size: 'compact-and-large' :", l67sValid({ size: "compact-and-large" }));

// The old impossible state — isCompact AND isLarge — cannot be written down at all now: `size`
// holds one value. That is the real gain. A validator can only reject a bad combination; a
// shape that cannot express it means there is nothing to reject and no resolution rule to
// document.

// 3. user={user} — both sides:
//    Right when the component is ABOUT a user: UserCard exists to display one, so a single prop
//    keeps the call sites short and the component free to show a new field without every caller
//    changing. Split it into fields the moment the component stops being about users — if you
//    want to reuse it for a supplier or a team, `name` and `subtitle` work and `user` does not,
//    because the prop's name and the object's shape have both become a lie.

**Common mistake:** "fixing" this by keeping the booleans and adding a resolution rule —
`isLarge` wins over `isCompact`, documented in a comment. That is a rule nobody at the call site
can see, and the impossible state still exists in the code. Removing the combination is a change
to the shape, not to the logic.

### LESSON 67 — Mini challenge

There is nothing to run here — the challenge is a review of your own Mini-project 3, and the
answers are specific to what you wrote. What a good review produces looks like this:

**A prop named for how it looks.** `<StatusDot green />` or `<Row highlighted />`. Rename to
meaning: `tone="active"`, `selected`. If you cannot name the meaning, that is itself the
finding — the component is being told about a pixel rather than about the data.

**A pair that can contradict.** `loading` and `error` passed as separate booleans is the most
likely one in this project, because topic 15 taught four states and props often flatten them
back into two booleans. What does your component render when both are true? If the answer is
"whichever `if` comes first", you have found an undocumented resolution rule. One `status` prop
with `"loading" | "error" | "empty" | "ready"` removes the question.

**A prop for a caller that never arrived.** Usually a `title` or `heading` prop with the same
string at the only call site, or a `variant` with one value ever used. Delete it and inline the
value; it costs nothing to add back when a second caller shows up with a real requirement.

Two notes on doing this well. Change **one** component and stop — the exercise is calibration,
not a refactor, and a finished project that works is worth more than a tidier one you broke on a
Sunday. And if all three components came out clean, take the result seriously rather than
hunting for something to fix: components extracted from working code tend to have honest prop
APIs, because every prop was added by a caller that actually existed. That is the habit this
lesson is trying to build.